# import libraries

In [20]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.cluster import KMeans, AgglomerativeClustering
import numpy as np
from sklearn.metrics import confusion_matrix, pairwise_distances_argmin, accuracy_score
from scipy.optimize import linear_sum_assignment
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
import pandas as pd

# Read dataset

In [2]:
iris = load_iris()

X = iris.data
y = iris.target

# Feature names and target names
feature_names = iris.feature_names
target_names = iris.target_names

print("Feature names:", feature_names)
print("Target names:", target_names)
print("First 5 samples:\n", X[:5])
print("First 5 labels:", y[:5])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)
print("y_train.shape:", y_train.shape)
print("y_test.shape:", y_test.shape)

Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']
First 5 samples:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
First 5 labels: [0 0 0 0 0]
X_train.shape: (105, 4)
X_test.shape: (45, 4)
y_train.shape: (105,)
y_test.shape: (45,)


# Train Kmeans clustering

In [3]:
# KMeans with k-means++ initialization
kmeans = KMeans(
    n_clusters=3,
    init="k-means++",   # this is the kmeans++ algorithm
    n_init=10,
    random_state=42
)

# Train
kmeans.fit(X_train)

# Results
kmeans_labels = kmeans.labels_
kmeans_centers = kmeans.cluster_centers_

print("Cluster labels (first 10):", kmeans_labels[:10])
print("Cluster centers:\n", kmeans_centers)

Cluster labels (first 10): [2 2 0 1 2 1 0 0 0 1]
Cluster centers:
 [[4.98857143 3.42571429 1.48571429 0.24      ]
 [6.97692308 3.14230769 5.84615385 2.11153846]
 [5.925      2.70909091 4.39545455 1.43863636]]


# calculate accuracy

In [5]:
def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    return cm[row_ind, col_ind].sum() / cm.sum()

kmeans_test_labels = kmeans.predict(X_test)
acc = clustering_accuracy(y_test, kmeans_test_labels)
print("KMeans accuracy:", acc)

KMeans accuracy: 0.8666666666666667


# Train hierarchical clustering and calculate its acc

In [6]:
hc = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"   # standard choice for Euclidean data
)

train_labels = hc.fit_predict(X_train)

# ---------- accuracy ----------
acc = clustering_accuracy(y_train, train_labels)

print("Hierarchical clustering accuracy:", acc)

Hierarchical clustering accuracy: 0.9047619047619048


In [7]:
centroids = np.vstack([
    X_train[train_labels == k].mean(axis=0)
    for k in np.unique(train_labels)
])

# pseudo-predict on new data
test_labels = pairwise_distances_argmin(X_test, centroids)

In [8]:
clustering_accuracy(y_test, test_labels)

np.float64(0.8888888888888888)

# Train mlp classifier

In [9]:
# generate pseudo labels
hc = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"   # standard choice for Euclidean data
)

pseudo_labels = hc.fit_predict(X)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, pseudo_labels, test_size=0.2, random_state=42
)



In [17]:
# Create pipeline (scaling is VERY important for MLP)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(early_stopping=False, tol=0, random_state=42))
])

# Define parameter distributions
param_dist = {
    'mlp__hidden_layer_sizes': [
        (50,), (100,),
        (10, 10), 
        (50, 50), (100, 50),
        (100, 100), 
        (50, 50, 50)
    ],
    'mlp__max_iter': randint(200, 1000),
    'mlp__alpha': uniform(0.0001, 0.01),
    'mlp__learning_rate_init': uniform(0.0001, 0.01)
}

# Randomized search
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,              # number of random combinations
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

# Fit
random_search.fit(X_train, y_train)
# Results
print("Best parameters:", random_search.best_params_)
print("Best CV score:", random_search.best_score_)

# Test score
test_score = random_search.score(X_test, y_test)
print("Test accuracy:", test_score)

Best parameters: {'mlp__alpha': np.float64(0.009485527090157502), 'mlp__hidden_layer_sizes': (100,), 'mlp__learning_rate_init': np.float64(0.0019182496720710062), 'mlp__max_iter': 476}
Best CV score: 0.9714285714285713
Test accuracy: 0.9111111111111111


c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (476) reached and the optimization hasn't converged yet.
  warnings.warn(


In [21]:
# Store results for analysis
results = []

for epoch in np.arange(100, 1000, 100):
    # Create pipeline with explicit naming
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=(100,),
            early_stopping=False,
            learning_rate_init=0.0019182496720710062,
            max_iter=epoch,
            tol=0,
            random_state=42,
            verbose=False  # Suppress convergence warnings if any
        ))
    ])
    
    # Fit the model
    pipeline.fit(X_train, y_train)
    
    # Make predictions
    y_pred = pipeline.predict(X_test)
    
    # Calculate metrics
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results.append({
        'epoch': epoch,
        'train_accuracy': train_score,
        'test_accuracy': test_score,
        'accuracy_diff': train_score - test_score
    })
    
    # Print formatted output
    print(f"Epochs: {epoch:4d} | Train Acc: {train_score:.4f} | Test Acc: {test_score:.4f} | Diff: {train_score - test_score:.4f}")

# Optional: Display summary
print("\n" + "="*60)
print("Summary of Results:")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Find best epoch based on test accuracy
best_result = max(results, key=lambda x: x['test_accuracy'])
print(f"\nBest epoch: {best_result['epoch']} with test accuracy: {best_result['test_accuracy']:.4f}")

c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Epochs:  100 | Train Acc: 0.9714 | Test Acc: 0.8667 | Diff: 0.1048
Epochs:  200 | Train Acc: 0.9810 | Test Acc: 0.9111 | Diff: 0.0698
Epochs:  300 | Train Acc: 0.9905 | Test Acc: 0.9111 | Diff: 0.0794


c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Epochs:  400 | Train Acc: 1.0000 | Test Acc: 0.9111 | Diff: 0.0889
Epochs:  500 | Train Acc: 1.0000 | Test Acc: 0.9111 | Diff: 0.0889


c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (700) reached and the optimization hasn't converged yet.
  warnings.warn(


Epochs:  600 | Train Acc: 1.0000 | Test Acc: 0.9333 | Diff: 0.0667
Epochs:  700 | Train Acc: 1.0000 | Test Acc: 0.9333 | Diff: 0.0667


c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (800) reached and the optimization hasn't converged yet.
  warnings.warn(


Epochs:  800 | Train Acc: 1.0000 | Test Acc: 0.9333 | Diff: 0.0667
Epochs:  900 | Train Acc: 1.0000 | Test Acc: 0.9333 | Diff: 0.0667

Summary of Results:
 epoch  train_accuracy  test_accuracy  accuracy_diff
   100        0.971429       0.866667       0.104762
   200        0.980952       0.911111       0.069841
   300        0.990476       0.911111       0.079365
   400        1.000000       0.911111       0.088889
   500        1.000000       0.911111       0.088889
   600        1.000000       0.933333       0.066667
   700        1.000000       0.933333       0.066667
   800        1.000000       0.933333       0.066667
   900        1.000000       0.933333       0.066667

Best epoch: 600 with test accuracy: 0.9333


c:\Anaconda\envs\torch-gpu\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (900) reached and the optimization hasn't converged yet.
  warnings.warn(
